# Reconstruction StockTwits / FinBERT

Ce notebook reconstruit les lignes brutes dans l'ordre, ajoute le label natif `Bullish`/`Bearish` au fichier FinBERT existant et bloque la comparaison si l'alignement n'est pas strictement vérifié. Aucune inférence n'est relancée.

In [7]:
import os, glob, re, html, gc, ast
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, f1_score, cohen_kappa_score

PROJECT = r"C:\Users\semy4\OneDrive\Bureau\Fintech_project"
RAW_DIR = os.path.join(PROJECT, "data", "raw", "source_data_trouve_stocktwits", "StockTwits_2020_2022_Raw")
PROCESSED_DIR = os.path.join(PROJECT, "data", "processed")
FINBERT_PATH = os.path.join(PROCESSED_DIR, "02_StockTwits_SCORED_2020_2022.csv")
OUTPUT_PATH = os.path.join(PROCESSED_DIR, "02_StockTwits_SCORED_2020_2022_WITH_NATIVE_LABEL.csv")
FOLDERS = ["AAPL_2020_2022", "AMZN2019-2022", "FB_2019_2022", "NVDA_2013_2022", "TSLA_2020_2022"]
VALID_TICKERS = {"AAPL", "AMZN", "META", "NVDA", "TSLA"}
SYMBOL_RE = re.compile(r"['\"]symbol['\"]\s*[:=]\s*['\"]([A-Z]+)['\"]")
CASHTAG_RE = re.compile(r"\$([A-Z]{1,5})\b")
assert os.path.isdir(RAW_DIR), RAW_DIR
assert os.path.isfile(FINBERT_PATH), FINBERT_PATH

## Reconstruction des colonnes utilisées pour l'inférence

In [8]:
def ticker_from_row(symbols, body):
    symbols = "" if pd.isna(symbols) else str(symbols)
    body = "" if pd.isna(body) else str(body)
    match = SYMBOL_RE.search(symbols) or CASHTAG_RE.search(body.upper())
    ticker = match.group(1) if match else ""
    ticker = "META" if ticker == "FB" else ticker
    return ticker if ticker in VALID_TICKERS else np.nan

def native_label(value):
    if pd.isna(value):
        return np.nan
    try:
        parsed = ast.literal_eval(str(value))
        sentiment = parsed.get("sentiment") if isinstance(parsed, dict) else None
        basic = sentiment.get("basic") if isinstance(sentiment, dict) else None
        if basic == "Bullish":
            return "positive"
        if basic == "Bearish":
            return "negative"
    except (ValueError, SyntaxError):
        pass
    return np.nan

def clean_text(value):
    value = "" if pd.isna(value) else html.unescape(str(value))
    value = re.sub(r"https?://\S+|www\.\S+|@\w+", "", value)
    return re.sub(r"\s+", " ", value).strip()

def read_raw(path):
    available = pd.read_csv(path, nrows=0).columns
    needed = [c for c in ["created_at", "body", "symbols", "entities"] if c in available]
    raw = pd.read_csv(path, usecols=needed)
    body = raw.get("body", pd.Series("", index=raw.index)).fillna("").astype(str)
    symbols = raw.get("symbols", pd.Series("", index=raw.index))
    entities = raw.get("entities", pd.Series(index=raw.index, dtype=object))
    out = pd.DataFrame({"Date": raw["created_at"], "Tweet": body, "Ticker": [ticker_from_row(s, b) for s, b in zip(symbols, body)], "native_label": entities.map(native_label)})
    return out.dropna(subset=["Date", "Ticker"])

paths = []
for folder in FOLDERS:
    paths.extend(glob.glob(os.path.join(RAW_DIR, folder, "*.csv")))
parts = [read_raw(path) for path in paths]
reconstructed = pd.concat(parts, ignore_index=True)
del parts
gc.collect()
reconstructed["Date"] = pd.to_datetime(reconstructed["Date"], utc=True, errors="coerce")
reconstructed = reconstructed.dropna(subset=["Date", "Tweet"])
# Le notebook 02 supprime les doublons sur les trois colonnes originales,
# avant le filtrage temporel. Le label natif ne fait pas partie de la clé.
reconstructed = reconstructed.drop_duplicates(subset=["Date", "Tweet", "Ticker"], keep="first").reset_index(drop=True)
reconstructed["Date_NY"] = reconstructed["Date"].dt.tz_convert("America/New_York")
reconstructed = reconstructed[reconstructed["Date_NY"].dt.year.isin([2020, 2021, 2022])].copy().reset_index(drop=True)
reconstructed["Jour"] = reconstructed["Date_NY"].dt.date.astype(str)
reconstructed["Heure_decimale"] = reconstructed["Date_NY"].dt.hour + reconstructed["Date_NY"].dt.minute / 60
reconstructed["Texte_Nettoye"] = reconstructed["Tweet"].map(clean_text)
reconstructed = reconstructed.drop(columns=["Date", "Date_NY"])
reconstructed = reconstructed.sort_values(by=["Ticker", "Jour", "Heure_decimale"]).reset_index(drop=True)
print(f"{len(paths)} fichiers, {len(reconstructed):,} lignes reconstruites")

362 fichiers, 3,711,333 lignes reconstruites


## Contrôle bloquant de l'alignement

In [9]:
finbert = pd.read_csv(FINBERT_PATH)
required = {"Tweet", "Ticker", "Jour", "Heure_decimale", "Texte_Nettoye", "FinBERT_Positive", "FinBERT_Negative", "FinBERT_Neutral"}
missing = required - set(finbert.columns)
if missing:
    raise ValueError(f"Colonnes absentes : {sorted(missing)}")
if len(reconstructed) != len(finbert):
    raise ValueError(f"Nombre de lignes différent : reconstruction={len(reconstructed):,}, FinBERT={len(finbert):,}")

def comparable(series):
    return series.map(lambda x: "" if pd.isna(x) else re.sub(r"\s+", " ", html.unescape(str(x))).strip()).to_numpy()

checks = {
    "Ticker": comparable(reconstructed["Ticker"]) == comparable(finbert["Ticker"]),
    "Tweet": comparable(reconstructed["Tweet"]) == comparable(finbert["Tweet"]),
    "Jour": comparable(reconstructed["Jour"]) == comparable(finbert["Jour"]),
    "Texte_Nettoye": comparable(reconstructed["Texte_Nettoye"]) == comparable(finbert["Texte_Nettoye"]),
    "Heure_decimale": np.isclose(reconstructed["Heure_decimale"], finbert["Heure_decimale"], equal_nan=True),
}
for name, result in checks.items():
    print(f"{name}: {int((~result).sum()):,} différences")
core_checks = {name: result for name, result in checks.items() if name != "Texte_Nettoye"}
if not all(result.all() for result in core_checks.values()):
    mismatch = ~(core_checks["Ticker"] & core_checks["Tweet"] & core_checks["Jour"] & core_checks["Heure_decimale"])
    display(pd.DataFrame({"ligne": np.flatnonzero(mismatch)[:20], "tweet_reconstruit": reconstructed.loc[mismatch, "Tweet"].head(20).to_numpy(), "tweet_finbert": finbert.loc[mismatch, "Tweet"].head(20).to_numpy(), "ticker_reconstruit": reconstructed.loc[mismatch, "Ticker"].head(20).to_numpy(), "ticker_finbert": finbert.loc[mismatch, "Ticker"].head(20).to_numpy()}))
    raise ValueError("Désalignement des observations détecté : comparaison annulée.")
if not checks["Texte_Nettoye"].all():
    print("Avertissement : Texte_Nettoye diffère sur", int((~checks["Texte_Nettoye"]).sum()), "lignes ; Tweet reste identique.")
    reconstructed["Texte_Nettoye"] = finbert["Texte_Nettoye"].to_numpy()
print("Alignement des observations validé.")

Ticker: 0 différences
Tweet: 0 différences
Jour: 0 différences
Texte_Nettoye: 1,111 différences
Heure_decimale: 0 différences
Avertissement : Texte_Nettoye diffère sur 1111 lignes ; Tweet reste identique.
Alignement des observations validé.


## Comparaison et export

In [15]:
comparison = finbert.copy()
comparison["Ground_Truth_Sentiment"] = reconstructed["native_label"].to_numpy()
comparison["native_label"] = comparison["Ground_Truth_Sentiment"]
prob_cols = ["FinBERT_Positive", "FinBERT_Negative", "FinBERT_Neutral"]
comparison["finbert_label"] = comparison[prob_cols].idxmax(axis=1).str.replace("FinBERT_", "", regex=False).str.lower()
comparison["finbert_confidence"] = comparison[prob_cols].max(axis=1)
print("Distribution ground truth :")
display(comparison["Ground_Truth_Sentiment"].value_counts(dropna=False).to_frame("n"))
labeled = comparison[comparison["Ground_Truth_Sentiment"].isin(["positive", "negative"])].copy()
labels = ["positive", "negative", "neutral"]
print(f"Lignes avec label natif : {len(labeled):,} / {len(comparison):,}")
print(classification_report(labeled["native_label"], labeled["finbert_label"], labels=labels, zero_division=0))
print("Balanced accuracy :", round(balanced_accuracy_score(labeled["native_label"], labeled["finbert_label"]), 4))
print("F1 macro :", round(f1_score(labeled["native_label"], labeled["finbert_label"], labels=labels, average="macro", zero_division=0), 4))
print("Cohen kappa :", round(cohen_kappa_score(labeled["native_label"], labeled["finbert_label"]), 4))
display(pd.DataFrame(confusion_matrix(labeled["native_label"], labeled["finbert_label"], labels=labels), index=labels, columns=labels))
comparison.to_csv(OUTPUT_PATH, index=False)
print("Export comparaison :", OUTPUT_PATH)

# Correction expérimentale one-hot, sans modifier les lignes sans ground truth.
CORRECTED_PATH = os.path.join(PROCESSED_DIR, "02_StockTwits_SCORED_2020_2022_GROUND_TRUTH_CORRECTED.csv")
corrected = finbert.copy()
native = reconstructed["native_label"].to_numpy()
labeled_mask = pd.notna(native)
positive_mask = labeled_mask & (native == "positive")
negative_mask = labeled_mask & (native == "negative")
corrected.loc[positive_mask, "FinBERT_Positive"] = 1.0
corrected.loc[positive_mask, "FinBERT_Negative"] = 0.0
corrected.loc[positive_mask, "FinBERT_Neutral"] = 0.0
corrected.loc[negative_mask, "FinBERT_Positive"] = 0.0
corrected.loc[negative_mask, "FinBERT_Negative"] = 1.0
corrected.loc[negative_mask, "FinBERT_Neutral"] = 0.0
corrected.to_csv(CORRECTED_PATH, index=False)
print(f"Lignes corrigées one-hot : {int(labeled_mask.sum()):,}")
print(f"Lignes conservées FinBERT : {int((~labeled_mask).sum()):,}")
print("Export test, structure identique à FinBERT :", CORRECTED_PATH)

Distribution ground truth :


,n
Ground_Truth_Sentiment,
NaN,1776462
positive,1448757
negative,486114


Lignes avec label natif : 1,934,871 / 3,711,333
              precision    recall  f1-score   support

    positive       0.87      0.15      0.25   1448757
    negative       0.49      0.11      0.18    486114
     neutral       0.00      0.00      0.00         0

    accuracy                           0.14   1934871
   macro avg       0.45      0.09      0.14   1934871
weighted avg       0.77      0.14      0.23   1934871



c:\Users\semy4\OneDrive\Bureau\Fintech_project\venv\lib\site-packages\sklearn\metrics\_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Balanced accuracy : 0.1293
F1 macro : 0.1444
Cohen kappa : 0.0325


,positive,negative,neutral
positive,213006,56403,1179348
negative,31852,54238,400024
neutral,0,0,0


Export comparaison : C:\Users\semy4\OneDrive\Bureau\Fintech_project\data\processed\02_StockTwits_SCORED_2020_2022_WITH_NATIVE_LABEL.csv
Lignes corrigées one-hot : 1,934,871
Lignes conservées FinBERT : 1,776,462
Export test, structure identique à FinBERT : C:\Users\semy4\OneDrive\Bureau\Fintech_project\data\processed\02_StockTwits_SCORED_2020_2022_GROUND_TRUTH_CORRECTED.csv


In [17]:
df=pd.read_csv("C:\\Users\\semy4\\OneDrive\\Bureau\\Fintech_project\\data\\processed\\02_StockTwits_SCORED_2020_2022_GROUND_TRUTH_CORRECTED.csv")

In [19]:
df.head(10)

,Tweet,Ticker,Jour,Heure_decimale,Texte_Nettoye,FinBERT_Positive,FinBERT_Negative,FinBERT_Neutral
0,$AAPL good things happening 2020 run trump and...,AAPL,2020-01-01,0.100000,$AAPL good things happening 2020 run trump and...,1.000000,0.000000e+00,0.000000
1,$AAPL Happy New Year amazing winning AAPL Bull...,AAPL,2020-01-01,0.133333,$AAPL Happy New Year amazing winning AAPL Bull...,1.000000,0.000000e+00,0.000000
2,Happy New Year :)\n$AAPL $TSLA $AMZN $SPY $B...,AAPL,2020-01-01,0.233333,Happy New Year :) $AAPL $TSLA $AMZN $SPY $BTC.X,1.000000,0.000000e+00,0.000000
3,@Taxes_R2_Damn_High my dad ended this year by...,AAPL,2020-01-01,0.316667,my dad ended this year by selling half his $aa...,0.000117,9.996450e-01,0.000238
4,$AAPL And to those using the tired and old del...,AAPL,2020-01-01,0.616667,$AAPL And to those using the tired and old del...,1.000000,0.000000e+00,0.000000
5,$AAPL **&quot;With massively more room to grow...,AAPL,2020-01-01,0.683333,"$AAPL **""With massively more room to grow"" not...",1.000000,0.000000e+00,0.000000
6,$AAPL gonna rip on Thursday for the new year,AAPL,2020-01-01,1.000000,$AAPL gonna rip on Thursday for the new year,1.000000,0.000000e+00,0.000000
7,$AAPL $MSFT $AMZN $PTI bulls and bears.. Happy...,AAPL,2020-01-01,1.066667,$AAPL $MSFT $AMZN $PTI bulls and bears.. Happy...,1.000000,0.000000e+00,0.000000
8,"Bullish investment portfolio: Apple($AAPL), In...",AAPL,2020-01-01,1.200000,"Bullish investment portfolio: Apple($AAPL), In...",0.999994,1.442521e-07,0.000006
9,"$AAPL can’t get AirPod pros anywhere, crazy stuff",AAPL,2020-01-01,2.066667,"$AAPL can’t get AirPod pros anywhere, crazy stuff",0.000002,7.777235e-04,0.999220


In [20]:
df.tail(10)

,Tweet,Ticker,Jour,Heure_decimale,Texte_Nettoye,FinBERT_Positive,FinBERT_Negative,FinBERT_Neutral
3711323,$TSLA Today was monthly close also as someone ...,TSLA,2022-02-28,20.666667,$TSLA Today was monthly close also as someone ...,1.000000e+00,0.000000,0.000000
3711324,$TSLA I guess by now. Nobody gives a flying b...,TSLA,2022-02-28,20.666667,$TSLA I guess by now. Nobody gives a flying bi...,1.000000e+00,0.000000,0.000000
3711325,"$LCID I have not tried shorting, but this feel...",TSLA,2022-02-28,20.716667,"$LCID I have not tried shorting, but this feel...",1.000000e+00,0.000000,0.000000
3711326,$TSLA can’t wait till tomorrow 🍻,TSLA,2022-02-28,20.733333,$TSLA can’t wait till tomorrow 🍻,5.822935e-06,0.000483,0.999511
3711327,$TSLA,TSLA,2022-02-28,20.733333,$TSLA,1.000000e+00,0.000000,0.000000
3711328,$TSLA 777,TSLA,2022-02-28,20.783333,$TSLA 777,0.000000e+00,1.000000,0.000000
3711329,$AMC $TSLA \n\nCan’t wait to buy a Tesla for o...,TSLA,2022-02-28,20.800000,$AMC $TSLA Can’t wait to buy a Tesla for overa...,1.000000e+00,0.000000,0.000000
3711330,$TSLA with future up now. any guess what will ...,TSLA,2022-02-28,20.800000,$TSLA with future up now. any guess what will ...,7.917394e-07,0.000115,0.999884
3711331,$TSLA what are the chances of Biden mentioning...,TSLA,2022-02-28,20.916667,$TSLA what are the chances of Biden mentioning...,1.000000e+00,0.000000,0.000000
3711332,$TSLA Shorts!! \n \n Tesla call put-ratio 1.4 ...,TSLA,2022-02-28,20.950000,"$TSLA Shorts!! Tesla call put-ratio 1.4 , with...",1.000000e+00,0.000000,0.000000


In [28]:
df["Texte_Nettoye"] = df["Texte_Nettoye"].fillna("")

In [29]:
df.isnull().sum()

Tweet               0
Ticker              0
Jour                0
Heure_decimale      0
Texte_Nettoye       0
FinBERT_Positive    0
FinBERT_Negative    0
FinBERT_Neutral     0
dtype: int64

In [30]:
print(df["Texte_Nettoye"].isna().sum())

0


In [31]:
df.head(10)

,Tweet,Ticker,Jour,Heure_decimale,Texte_Nettoye,FinBERT_Positive,FinBERT_Negative,FinBERT_Neutral
0,$AAPL good things happening 2020 run trump and...,AAPL,2020-01-01,0.100000,$AAPL good things happening 2020 run trump and...,1.000000,0.000000e+00,0.000000
1,$AAPL Happy New Year amazing winning AAPL Bull...,AAPL,2020-01-01,0.133333,$AAPL Happy New Year amazing winning AAPL Bull...,1.000000,0.000000e+00,0.000000
2,Happy New Year :)\n$AAPL $TSLA $AMZN $SPY $B...,AAPL,2020-01-01,0.233333,Happy New Year :) $AAPL $TSLA $AMZN $SPY $BTC.X,1.000000,0.000000e+00,0.000000
3,@Taxes_R2_Damn_High my dad ended this year by...,AAPL,2020-01-01,0.316667,my dad ended this year by selling half his $aa...,0.000117,9.996450e-01,0.000238
4,$AAPL And to those using the tired and old del...,AAPL,2020-01-01,0.616667,$AAPL And to those using the tired and old del...,1.000000,0.000000e+00,0.000000
5,$AAPL **&quot;With massively more room to grow...,AAPL,2020-01-01,0.683333,"$AAPL **""With massively more room to grow"" not...",1.000000,0.000000e+00,0.000000
6,$AAPL gonna rip on Thursday for the new year,AAPL,2020-01-01,1.000000,$AAPL gonna rip on Thursday for the new year,1.000000,0.000000e+00,0.000000
7,$AAPL $MSFT $AMZN $PTI bulls and bears.. Happy...,AAPL,2020-01-01,1.066667,$AAPL $MSFT $AMZN $PTI bulls and bears.. Happy...,1.000000,0.000000e+00,0.000000
8,"Bullish investment portfolio: Apple($AAPL), In...",AAPL,2020-01-01,1.200000,"Bullish investment portfolio: Apple($AAPL), In...",0.999994,1.442521e-07,0.000006
9,"$AAPL can’t get AirPod pros anywhere, crazy stuff",AAPL,2020-01-01,2.066667,"$AAPL can’t get AirPod pros anywhere, crazy stuff",0.000002,7.777235e-04,0.999220


In [32]:
df.to_csv("C:\\Users\\semy4\\OneDrive\\Bureau\\Fintech_project\\data\\processed\\02_StockTwits_SCORED_2020_2022_GROUND_TRUTH_CORRECTED_v1.csv", index=False)